In [ ]:
# Teaching simulator: mimics the dynamic-index-loading pattern
# Production swap-in: a real faiss.read_index() call on a .index file.
 
import os
import numpy as np
from typing import Dict, List, Tuple
 
class FakeFAISSRouter:
    '''
    Simulates one FAISS .index file per tenant, kept on disk, loaded on demand.
    In production: faiss.read_index(f'/mnt/indexes/{tenant_id}.index').
    Here: an in-memory dict keyed by tenant_id -> list of (vec, text, doc_id).
    '''
 
    def __init__(self):
        # Simulates the /mnt/indexes/ directory
        self._disk: Dict[str, List[Tuple[np.ndarray, str, str]]] = {}
        self._loaded_tenant: str | None = None
        self._loaded_index: List[Tuple[np.ndarray, str, str]] | None = None
        self.load_calls = 0
        self.unload_calls = 0
 
    def upsert(self, tenant_id: str, vec: np.ndarray, text: str, doc_id: str):
        '''Write a vector into tenant_id's OWN index file. Never shared.'''
        self._disk.setdefault(tenant_id, []).append((vec, text, doc_id))
 
    def load(self, tenant_id: str):
        '''Load a tenant's index into memory; unload any previous one.'''
        if self._loaded_tenant == tenant_id:
            return
        if self._loaded_tenant is not None:
            self._loaded_index = None
            self.unload_calls += 1
        self._loaded_index = self._disk.get(tenant_id, [])
        self._loaded_tenant = tenant_id
        self.load_calls += 1
 
    def search(self, tenant_id: str, query_vec: np.ndarray, k: int = 3):
        '''Search ONLY the loaded tenant's index. Hard isolation.'''
        self.load(tenant_id)
        assert self._loaded_tenant == tenant_id, 'router desync'
        scored = []
        for vec, text, doc_id in self._loaded_index or []:
            sim = float(np.dot(vec, query_vec))
            scored.append((sim, text, doc_id))
        scored.sort(key=lambda x: -x[0])
        return scored[:k]



In [ ]:
# REFERENCE ONLY (docx codeBlock, not executed): real FAISS per-tenant loader.
# pip install faiss-cpu
 
import faiss, os, threading
 
_INDEX_DIR = '/mnt/indexes'
_cache: dict[str, faiss.Index] = {}
_lock = threading.Lock()
 
def load_tenant_index(tenant_id: str) -> faiss.Index:
    '''Thread-safe per-tenant loader. Evicts via LRU in production.'''
    with _lock:
        if tenant_id in _cache:
            return _cache[tenant_id]
        path = os.path.join(_INDEX_DIR, f'{tenant_id}.index')
        if not os.path.exists(path):
            raise FileNotFoundError(f'no index for tenant {tenant_id}')
        idx = faiss.read_index(path)
        _cache[tenant_id] = idx
        return idx
 
def search(tenant_id: str, query_vec, k: int = 3):
    idx = load_tenant_index(tenant_id)
    return idx.search(query_vec.reshape(1, -1), k)



FakeMemorySaverCompositeKey: teaching simulator

In [ ]:
# Teaching simulator for LangGraph's SqliteSaver with COMPOSITE keys.
# Real class: langgraph.checkpoint.sqlite.SqliteSaver
# Real fix:  override .get_tuple / .put to include tenant_id in WHERE.
 
import sqlite3
import json
from typing import Optional, Dict, Any
 
class FakeMemorySaverCompositeKey:
    '''
    Stores checkpoints keyed by (tenant_id, thread_id) instead of thread_id alone.
    Prevents the classic memory-bleed CVE where User@B guesses User@A's thread_id.
    '''
 
    def __init__(self, db_path: str = ':memory:'):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute('''
            CREATE TABLE IF NOT EXISTS checkpoints (
                tenant_id TEXT NOT NULL,
                thread_id TEXT NOT NULL,
                step INTEGER NOT NULL,
                state_json TEXT NOT NULL,
                PRIMARY KEY (tenant_id, thread_id, step)
            )
        ''')
        self.conn.commit()
 
    def put(self, tenant_id: str, thread_id: str, step: int, state: Dict[str, Any]):
        self.conn.execute(
            'INSERT OR REPLACE INTO checkpoints VALUES (?,?,?,?)',
            (tenant_id, thread_id, step, json.dumps(state)),
        )
        self.conn.commit()
 
    def get_tuple(self, tenant_id: str, thread_id: str) -> Optional[Dict[str, Any]]:
        '''The COMPOSITE filter. Both fields must match. No fallback to thread_id alone.'''
        cur = self.conn.execute(
            'SELECT step, state_json FROM checkpoints '
            'WHERE tenant_id = ? AND thread_id = ? '
            'ORDER BY step DESC LIMIT 1',
            (tenant_id, thread_id),
        )
        row = cur.fetchone()
        if row is None:
            return None
        return {'step': row[0], 'state': json.loads(row[1])}



LangGraph node reading tenant_id from state and calling Gemini



# LangGraph node that calls Gemini with tenant-scoped context.
# Uses google.generativeai only; no OpenAI/Anthropic/LangChain/LiteLLM.
 
import google.generativeai as genai
 
MODEL_UTILITY = 'gemini-2.5-flash'
MODEL_FRONTIER = 'gemini-2.5-pro'
 
def answer_node(state: dict, router: 'FakeFAISSRouter') -> dict:
    '''
    state must contain:
      - 'tenant_id'  (injected by gateway middleware)
      - 'thread_id'  (injected by gateway middleware)
      - 'question'   (user text)
    '''
    tenant_id = state['tenant_id']
    question = state['question']
    # Fake embedding for teaching; production uses text-embedding-004.
    import numpy as np
    qvec = np.random.RandomState(hash(question) & 0xFFFF).rand(8)
    qvec = qvec / np.linalg.norm(qvec)
    hits = router.search(tenant_id, qvec, k=3)
    ctx = '\n'.join(f'- {h[1]}' for h in hits)
    prompt = (
        f'You are a tenant-scoped assistant for tenant {tenant_id}.\n'
        f'ONLY use the context below. If empty, say you have no data.\n\n'
        f'Context:\n{ctx}\n\nQuestion: {question}'
    )
    model = genai.GenerativeModel(MODEL_UTILITY)
    resp = model.generate_content(prompt)
    return {**state, 'answer': resp.text, 'retrieved': [h[2] for h in hits]}



Gemini-only runtime configuration (notebook header)



# Gemini-only runtime wiring (notebook top-of-file).
import google.generativeai as genai
from google.colab import userdata
 
api_key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=api_key)
 
MODEL_UTILITY = 'gemini-2.5-flash'    # cheap, fast, for per-turn agent calls
MODEL_FRONTIER = 'gemini-2.5-pro'     # higher quality, for planner / reviewer steps
 
print('Gemini configured. Utility =', MODEL_UTILITY, '| Frontier =', MODEL_FRONTIER)



REFERENCE: real Postgres RLS setup with tenant_isolation policy



-- REFERENCE ONLY (docx codeBlock, not executed): real Postgres RLS setup.
-- docker run -e POSTGRES_PASSWORD=dev -p 5432:5432 postgres:16
 
CREATE TABLE support_tickets (
    ticket_id   TEXT PRIMARY KEY,
    tenant_id   TEXT NOT NULL,
    subject     TEXT NOT NULL,
    body        TEXT NOT NULL
);
 
ALTER TABLE support_tickets ENABLE ROW LEVEL SECURITY;
ALTER TABLE support_tickets FORCE ROW LEVEL SECURITY;   -- applies to table owner too
 
CREATE POLICY tenant_isolation ON support_tickets
    USING (tenant_id = current_setting('app.current_tenant', true))
    WITH CHECK (tenant_id = current_setting('app.current_tenant', true));
 
-- Each API request sets the GUC once per connection:
SET LOCAL app.current_tenant = 'tenant_A';
 
-- Agent hallucinates:  SELECT * FROM support_tickets;
-- Engine executes:     SELECT * FROM support_tickets WHERE tenant_id = 'tenant_A';



FakePostgresRLS: teaching simulator with set_current_tenant + select



# Teaching simulator for Postgres Row-Level Security.
# Production swap-in: real Postgres with RLS policies; this fake mirrors semantics.
 
from typing import List, Dict, Any, Optional
 
class FakePostgresRLS:
    '''
    Simulates a Postgres table with an RLS policy:
        CREATE POLICY tenant_isolation ON support_tickets
        USING (tenant_id = current_setting('app.current_tenant'));
    No matter what WHERE clause the agent hallucinates, the engine
    conjoins `AND tenant_id = <current_tenant>` automatically.
    '''
 
    def __init__(self):
        self._rows: List[Dict[str, Any]] = []
        self._current_tenant: Optional[str] = None
 
    def set_current_tenant(self, tenant_id: str):
        '''Backend middleware calls this once per connection, per request.'''
        self._current_tenant = tenant_id
 
    def insert(self, tenant_id: str, ticket_id: str, subject: str, body: str):
        self._rows.append({
            'tenant_id': tenant_id,
            'ticket_id': ticket_id,
            'subject': subject,
            'body': body,
        })
 
    def select_all_support_tickets(self) -> List[Dict[str, Any]]:
        '''
        The agent thinks it is running `SELECT * FROM support_tickets`.
        The RLS policy silently appends `WHERE tenant_id = <current>`.
        If current_tenant is None, the query returns zero rows (deny-by-default).
        '''
        if self._current_tenant is None:
            return []
        return [r for r in self._rows if r['tenant_id'] == self._current_tenant]



exercise: agent hallucinates unfiltered SELECT; RLS still isolates



# RLS probe. Agent hallucinates an unfiltered SELECT; engine still isolates.
 
db = FakePostgresRLS()
db.insert('tenant_A', 'T-1', 'wire transfer failing', 'error 4000 on SWIFT code XYZ')
db.insert('tenant_A', 'T-2', 'ATM offline', 'branch 12 ATM network down')
db.insert('tenant_B', 'T-3', 'MRI scheduling bug', 'calendar double-booking')
 
# Backend middleware sets the GUC before the agent runs.
db.set_current_tenant('tenant_A')
 
# Agent's hallucinated SQL: SELECT * FROM support_tickets
rows = db.select_all_support_tickets()
print('As tenant_A, agent sees:')
for r in rows:
    print(' ', r['ticket_id'], '-', r['subject'])
 
# Switch connection to tenant_B
db.set_current_tenant('tenant_B')
rows = db.select_all_support_tickets()
print('\\nAs tenant_B, same query returns:')
for r in rows:
    print(' ', r['ticket_id'], '-', r['subject'])

